# BIO-001 cellulose surface virtual experiment

This notebook demonstrates the first insoluble cellulose-like substrate virtual experiment in FungMod. It uses registry IDs, runs modelability preflight, executes an exploratory surface-catalysis ensemble, and reads the publication-oriented output tables back from disk.

This is an enzyme-mediated insoluble cellulose surface-degradation pilot. It is not a whole-fungus growth model. It does not include fungal secretion, uptake, biomass growth, oxygen limitation, or full lignocellulose structure. Any exploratory parameter ranges are user-supplied unless explicitly sourced.

In [ ]:
from pathlib import Path
import csv
import os
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data_registry" / "registry_index.yml").exists():
    ROOT = Path("..").resolve()

src_path = ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from fungal_model.api import VirtualExperiment
from fungal_model.registry import load_registry


In [ ]:
FUNGUS_ID = "generic_cellulase_source"
SUBSTRATE_ID = "cellulose_film_generic"
ENVIRONMENT_ID = "bio001_cellulose_surface_pilot_environment"
REGISTRY_INDEX = ROOT / "data_registry" / "registry_index.yml"

OUTPUT_ROOT = Path(os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", str(ROOT / "outputs")))
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_ROOT / "mplconfig"))
screen_output_dir = OUTPUT_ROOT / "bio001_cellulose_surface_virtual_experiment"


In [ ]:
registry = load_registry(REGISTRY_INDEX)
study = VirtualExperiment.from_registry(
    fungi=[FUNGUS_ID],
    substrates=[SUBSTRATE_ID],
    environments=[ENVIRONMENT_ID],
    registry=registry,
)

scientific_preflight = study.preflight(mode="scientific")[0]
exploratory_preflight = study.preflight(mode="exploratory")[0]

{
    "scientific_status": scientific_preflight.status,
    "exploratory_status": exploratory_preflight.status,
    "required_processes": exploratory_preflight.required_processes,
    "required_parameters": exploratory_preflight.required_parameters,
}


In [ ]:
virtual_result = study.simulate(
    mode="exploratory",
    n_samples=8,
    seed=21,
    output_dir=screen_output_dir,
    quicklook=True,
)
screen = virtual_result.screen_result
screen_output_dir = Path(virtual_result.output_directory)

{
    "simulated_status": screen.case_results[0].modelability_report.status,
    "sample_count": len(screen.case_results[0].samples),
    "output_dir": str(screen_output_dir),
    "quicklook_paths": list(virtual_result.quicklook_paths),
}


In [ ]:
def read_rows(path):
    with Path(path).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

time_series_long = read_rows(screen_output_dir / "time_series_long.csv")
final_metrics = read_rows(screen_output_dir / "final_metrics.csv")
threshold_times = read_rows(screen_output_dir / "threshold_times.csv")

time_series_long[:8]


In [ ]:
final_metrics[:12]


In [ ]:
threshold_times


In [ ]:
import matplotlib.pyplot as plt

def numeric(value):
    return None if value in {None, ""} else float(value)

by_time = {}
for row in time_series_long:
    if row["state"] != "solid_substrate_degraded_fraction":
        continue
    time = numeric(row["time"])
    value = numeric(row["value"])
    if time is None or value is None:
        continue
    by_time.setdefault(time, []).append(value)

times = sorted(by_time)
lower = [min(by_time[time]) for time in times]
upper = [max(by_time[time]) for time in times]
median = [sorted(by_time[time])[len(by_time[time]) // 2] for time in times]

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(times, lower, upper, alpha=0.2, label="sample range")
ax.plot(times, median, label="median sample")
ax.set_xlabel("time (second)")
ax.set_ylabel("solid substrate degraded fraction")
ax.legend()
fig.tight_layout()
uncertainty_plot_path = screen_output_dir / "figures" / "bio001_degradation_uncertainty_band.png"
fig.savefig(uncertainty_plot_path, dpi=200)
plt.close(fig)

list(virtual_result.quicklook_paths) + [str(uncertainty_plot_path)]


In [ ]:
limitations_table = read_rows(screen_output_dir / "limitations_table.csv")
[row for row in limitations_table if row["category"] in {"not_modelled", "surface_accessibility", "exploratory_prior"}]
